# KeywordFinderAgent 動作確認

## 概要

**KeywordFinderAgent** は、指定されたカテゴリとシードキーワードをもとに、関連キーワードを自動で調査・発見するエージェントです。

### ワークフロー

1. **Plan（計画）** — シードキーワードから調査計画を立案
2. **Execute（実行）** — 検索ツールを使ってキーワードを収集
3. **Reflect（内省）** — 収集結果を評価し、追加調査が必要か判定
4. **Integrate（統合）** — 最終的なキーワードリストと分析サマリーを生成

### 必要な環境変数

- `OPENAI_API_KEY` — OpenAI API キー
- `SERPAPI_API_KEY` — SerpAPI キー（検索ツール用）

これらは `tools/.env` に設定してください。

In [1]:
import subprocess
import sys
from pathlib import Path

# tools/ ディレクトリを特定
notebook_dir = Path.cwd() if "__file__" not in dir() else Path(__file__).parent
# notebook が tools/src/agents/keyword_finder/notebook/ にある場合
# tools/ は 4階層上
for candidate in [
    notebook_dir.parents[3],  # src/agents/keyword_finder/notebook -> tools/
    notebook_dir.parents[4],  # もう1階層上
    Path.cwd(),
]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
        tools_dir = str(candidate)
        break
else:
    tools_dir = str(Path.cwd())

if tools_dir not in sys.path:
    sys.path.insert(0, tools_dir)

# python-dotenv がなければインストール
try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv"])
    from dotenv import load_dotenv

# tools/.env から環境変数を読み込み
env_path = Path(tools_dir) / ".env"
load_dotenv(env_path)
print(f"Python: {sys.executable}")
print(f"tools dir: {tools_dir}")
print(f".env loaded: {env_path.exists()}")

Python: /Users/tomo/Desktop/workspace/writing-project/.venv/bin/python
tools dir: /Users/tomo/Desktop/workspace/writing-project/writing/tools
.env loaded: True


## 入力データの説明

`KeywordSearchInput` は以下のフィールドを持ちます。

| フィールド | 型 | 説明 |
|---|---|---|
| `category` | `"資産形成"` \| `"健康"` \| `"エンジニア"` | 検索対象の分野 |
| `seed_keywords` | `list[str]` | シードキーワードのリスト（1つ以上） |
| `depth` | `int` (1-3) | 深掘りレベル（デフォルト: 2） |

In [2]:
from src.agents.keyword_finder.schemas import KeywordSearchInput

input_data = KeywordSearchInput(
    category="資産形成",
    seed_keywords=["新NISA", "投資信託"],
    depth=2,
)

print(input_data.model_dump_json(indent=2))

{
  "category": "資産形成",
  "seed_keywords": [
    "新NISA",
    "投資信託"
  ],
  "depth": 2
}


## エージェント実行

`KeywordFinderAgent` をインスタンス化し、`run()` メソッドで実行します。

> **注意**: 実行すると OpenAI API と SerpAPI への呼び出しが発生します。API キーが正しく設定されていることを確認してください。

In [3]:
from src.agents.keyword_finder.agent import KeywordFinderAgent

agent = KeywordFinderAgent()
output = agent.run(input_data)

print(f"カテゴリ: {output.category}")
print(f"キーワード数: {len(output.results)}")

2026-01-30 01:19:53 | INFO     | src.agents.keyword_finder.nodes | 計画立案を開始
2026-01-30 01:19:57 | INFO     | src.agents.keyword_finder.nodes | 計画立案完了: 5個のサブタスク
2026-01-30 01:19:57 | INFO     | src.agents.keyword_finder.nodes | ツール実行を開始
2026-01-30 01:19:57 | INFO     | src.agents.keyword_finder.nodes | サブタスク実行: get_related_keywordsを使用して、シードキーワード「新NISA」に関連するキーワードを取得する。
2026-01-30 01:19:58 | INFO     | src.agents.keyword_finder.nodes | ツール呼び出し: get_related_keywords({'keyword': '新NISA'})
2026-01-30 01:19:58 | INFO     | src.tools.related_keywords | 関連キーワード取得開始: keyword='新NISA'
2026-01-30 01:19:58 | INFO     | src.tools.related_keywords | 関連キーワード取得完了: 10件
2026-01-30 01:19:58 | INFO     | src.agents.keyword_finder.nodes | サブタスク実行: get_related_keywordsを使用して、シードキーワード「投資信託」に関連するキーワードを取得する。
2026-01-30 01:19:59 | INFO     | src.agents.keyword_finder.nodes | ツール呼び出し: get_related_keywords({'keyword': '投資信託'})
2026-01-30 01:19:59 | INFO     | src.tools.related_keywords | 関連キーワード取得開始: keyword='投資信託'
2

## 結果の確認

`KeywordSearchOutput` の構造:

| フィールド | 型 | 説明 |
|---|---|---|
| `category` | `str` | 検索した分野 |
| `seed_keywords` | `list[str]` | 使用したシードキーワード |
| `results` | `list[KeywordResult]` | 検索結果のリスト |
| `summary` | `str` | 分析サマリー |

各 `KeywordResult` には `keyword`, `search_volume`, `competition`, `relevance_score`, `suggested_topics` が含まれます。

In [4]:
# サマリー表示
print("=" * 60)
print(f"カテゴリ: {output.category}")
print(f"シードキーワード: {', '.join(output.seed_keywords)}")
print("=" * 60)
print(f"\n{output.summary}\n")

# 各キーワード結果を表示
print("-" * 60)
print(f"{'キーワード':<20} {'検索Vol':>8} {'競合':>4} {'関連度':>6}")
print("-" * 60)

for r in output.results:
    vol = str(r.search_volume) if r.search_volume is not None else "-"
    print(f"{r.keyword:<20} {vol:>8} {r.competition:>4} {r.relevance_score:>6.2f}")
    if r.suggested_topics:
        for topic in r.suggested_topics:
            print(f"  -> {topic}")

print("-" * 60)
print(f"合計: {len(output.results)} 件")

カテゴリ: 資産形成
シードキーワード: 新NISA, 投資信託

新NISAおよび投資信託に関連するキーワードを分析した結果、いくつかの記事テーマ案が浮かび上がりました。特に、投資信託に関するシミュレーションやランキング、新NISAの詳細やデメリットについての記事は高い関連性を持ち、読者の興味を引く可能性が高いです。また、競合状況としては投資信託に関するキーワードが高まっているため、独自の視点や具体的な情報を提供することで競争を勝ち抜くことができるでしょう。全体として、関連度が高く、競合度も適切なキーワードが揃っています。

------------------------------------------------------------
キーワード                   検索Vol   競合    関連度
------------------------------------------------------------
投資信託 シミュレーション               -    中   0.80
  -> 投資信託シミュレーションの利用方法
  -> 実際の投資信託シミュレーション結果
  -> シミュレーションを使った資産形成の効果
投資信託 ランキング                  -    高   0.90
  -> 2023年の投資信託ランキング
  -> 初心者向け投資信託ランキング
  -> 人気の投資信託の特徴
新nisa 成長投資枠とは               -    中   0.85
  -> 新NISAの成長投資枠の詳細
  -> 成長投資枠のメリットとデメリット
  -> 新NISAを活用した資産形成の戦略
新nisa 積立投資枠                 -    中   0.80
  -> 新NISAの積立投資枠の特徴
  -> 積立投資枠の活用法
  -> 新NISAでの積立投資の利点
新nisa デメリット                 -    中   0.75
  -> 新NISAのデメリットとその対策
  -> 新NISAを利用する際の注意点
  -> 投資信託との比較
投資信託 おすすめ                   -    高   0.85
  -> おすすめの投資